In [27]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [6]:
adp = pd.read_csv("adp_full_2025_with_prev_obj.csv")
adp = adp.drop(columns=['Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'])
print(adp.columns)

Index(['Rank', 'Player', 'Team', 'Bye', 'POS', 'ESPN', 'Sleeper', 'CBS', 'NFL',
       'RTSports', 'Fantrax', 'AVG', 'Real-Time (?)', 'gsis_id', 'position',
       'season', 'availability', 'age', 'rookie_season', 'snap_share',
       'fan_pts_var', 'apy_cap_pct', 'draft_pick', 'pick_value',
       'prev_year_games', 'prev_year_completions', 'prev_year_passing_yards',
       'prev_year_passing_tds', 'prev_year_carries', 'prev_year_rushing_yards',
       'prev_year_rushing_tds', 'prev_year_receptions',
       'prev_year_receiving_yards', 'prev_year_receiving_tds',
       'prev_year_turnovers'],
      dtype='object')


In [32]:
cols = ['fan_pts_var','availability','prev_year_games', 'prev_year_completions', 'prev_year_passing_yards',
       'prev_year_passing_tds', 'prev_year_carries', 'prev_year_rushing_yards',
       'prev_year_rushing_tds', 'prev_year_receptions',
       'prev_year_receiving_yards', 'prev_year_receiving_tds',
       'prev_year_turnovers']
for col in cols:
    adp[col] = adp[col].replace({"--": 0})
    adp[col] = pd.to_numeric(adp[col], errors='coerce').fillna(0)
adp['half_ppr_points'] = (0.1*adp['prev_year_receiving_yards'] + 0.1*adp['prev_year_rushing_yards'] + 0.04*adp['prev_year_passing_yards'] +
                6*adp['prev_year_rushing_tds'] + 6*adp['prev_year_receiving_tds'] + 4*adp['prev_year_passing_tds'] +
                0.5*adp['prev_year_receptions'] - 2*adp['prev_year_turnovers'])
adp['full_ppr_points'] = (0.1*adp['prev_year_receiving_yards'] + 0.1*adp['prev_year_rushing_yards'] + 0.04*adp['prev_year_passing_yards'] +
                6*adp['prev_year_rushing_tds'] + 6*adp['prev_year_receiving_tds'] + 4*adp['prev_year_passing_tds'] +
                1*adp['prev_year_receptions'] - 2*adp['prev_year_turnovers'])
print(adp.head())

   Rank            Player Team   Bye  POS  ESPN  Sleeper  CBS  NFL  RTSports  \
0     1     Ja'Marr Chase  CIN  10.0  WR1   1.0      1.0  1.0  1.0       1.0   
1     2    Bijan Robinson  ATL   5.0  RB1   2.0      3.0  2.0  2.0       2.0   
2     3    Saquon Barkley  PHI   9.0  RB2   3.0      2.0  3.0  4.0       4.0   
3     4      Jahmyr Gibbs  DET   8.0  RB3   5.0      4.0  4.0  6.0       3.0   
4     5  Justin Jefferson  MIN   6.0  WR2   4.0      5.0  6.0  5.0       6.0   

   ...  prev_year_rushing_tds  prev_year_receptions prev_year_receiving_yards  \
0  ...                      0                   127                    1708.0   
1  ...                     14                    61                     431.0   
2  ...                     13                    33                     278.0   
3  ...                     16                    52                     497.0   
4  ...                      0                   103                    1533.0   

  prev_year_receiving_tds prev_y

In [54]:
adp = adp[adp['position'].isin(['RB','WR','TE','CB'])]
adp['std_dev'] = pd.to_numeric(np.sqrt(adp['fan_pts_var']))
adp['year_availability'] = pd.to_numeric(adp['prev_year_games']/17)
adp['fan_per_game'] = pd.to_numeric(adp['full_ppr_points']/adp['prev_year_games'])
adp['rookie_season'] = pd.to_numeric(adp['rookie_season'].replace({"--":2023,"D/ST":2023}))
adp['experience'] = 2025 - adp['rookie_season']
adp.loc[adp['position'] == 'QB', 'experience'] = adp.loc[adp['position'] == 'QB', 'experience'] / 3
adp['experience'] = pd.to_numeric(adp['experience'])
cols = ['fan_per_game', 'year_availability', 'std_dev', 'availability', 'experience']
scaler = MinMaxScaler()
adp_scaled = adp.copy()
adp_scaled[cols] = scaler.fit_transform(adp[cols])

C:\Users\rille\AppData\Local\Temp\ipykernel_14100\2130470957.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adp['std_dev'] = pd.to_numeric(np.sqrt(adp['fan_pts_var']))
C:\Users\rille\AppData\Local\Temp\ipykernel_14100\2130470957.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adp['year_availability'] = pd.to_numeric(adp['prev_year_games']/17)
C:\Users\rille\AppData\Local\Temp\ipykernel_14100\2130470957.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.

In [58]:
adp_scaled['security_score'] = (adp_scaled['fan_per_game']*0.3
                                + 0.3*adp_scaled['std_dev'] + 0.01*adp_scaled['experience']
                                + 0.19*adp_scaled['year_availability'] + 0.2*adp_scaled['availability'])
adp_scaled['round'] = adp_scaled['Rank']//12 + 1
print(adp_scaled[['Rank','round','Player','security_score']].sort_values(by='security_score', ascending=False).head(15))

     Rank  round              Player  security_score
0       1      1       Ja'Marr Chase        0.977961
2       3      1      Saquon Barkley        0.883206
3       4      1        Jahmyr Gibbs        0.858772
7       8      1   Amon-Ra St. Brown        0.835033
9      10      1       Derrick Henry        0.824487
17     18      2        Drake London        0.808495
38     39      4          Mike Evans        0.805617
39     40      4       Davante Adams        0.803091
1       2      1      Bijan Robinson        0.799364
14     15      2       De'Von Achane        0.795240
4       5      1    Justin Jefferson        0.794983
104   105      9      Jauan Jennings        0.792508
13     14      2    Brian Thomas Jr.        0.788076
32     33      3  Jaxon Smith-Njigba        0.785776
28     29      3         Tee Higgins        0.784045


In [70]:
top_security = adp_scaled.sort_values(['round', 'security_score'], ascending=[True, False]).groupby('round')['security_score'].nlargest(2).reset_index()
max_by_round = adp_scaled.merge(top_security[['round', 'level_1']], left_index=True, right_on='level_1', how='inner').drop(columns=['level_1'])
max_by_round = max_by_round[max_by_round['round_x']<=10]
print(max_by_round[['Rank','Player','round_x','security_score']].sort_values(by=['round_x','security_score'], ascending=[True, False]))

    Rank              Player  round_x  security_score
0      1       Ja'Marr Chase        1        0.977961
1      3      Saquon Barkley        1        0.883206
2     18        Drake London        2        0.808495
3     15       De'Von Achane        2        0.795240
4     33  Jaxon Smith-Njigba        3        0.785776
5     29         Tee Higgins        3        0.784045
6     39          Mike Evans        4        0.805617
7     40       Davante Adams        4        0.803091
8     48            DJ Moore        5        0.733585
9     55       DeVonta Smith        5        0.728718
10    61        Travis Kelce        6        0.706519
11    68     Aaron Jones Sr.        6        0.682272
12    74         Jerry Jeudy        7        0.770381
13    77    Tyrone Tracy Jr.        7        0.692844
14    84           Joe Mixon        8        0.783659
15    91      Jordan Addison        8        0.779770
16   105      Jauan Jennings        9        0.792508
17   107        Tucker Kraft